# Embedding Sanity Check

This notebook poke-tests the two embedding spaces produced by `01_train_word2vec.py`, `02_encode_sbert.py`, and `03_pool_ingredients.py`:

1. **Ingredient nearest neighbors** (Word2Vec) — does basil cluster with other herbs?
2. **Recipe nearest neighbors** (SBERT) — does a Thai noodle dish find other Thai noodle dishes?
3. **Vector analogies** (Word2Vec) — does the embedding space have meaningful directions?
4. **SBERT vs ingredient-pooled** — which space clusters cuisine better?

If any of section 1's results look obviously wrong (e.g. nearest neighbor of `basil` is `1/2`), the bug is upstream in Agent 1's vocab and needs to be flagged before Agent 3 runs anything.

## 0. Load everything

In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

# EMB_DIR = Path("./embeddings")
EMB_DIR = Path("./")
DATA_PATH = Path("../data/processed/recipes_clean.pkl")

df = pd.read_pickle(DATA_PATH)
print(f"Loaded {len(df)} recipes")

w2v_model = Word2Vec.load(str(EMB_DIR / "word2vec.model"))
ingredient_vectors = np.load(EMB_DIR / "ingredient_vectors.npy")
with open(EMB_DIR / "ingredient_id_map.json", encoding="utf-8") as f:
    ingredient_id_map = json.load(f)

recipe_vectors = np.load(EMB_DIR / "recipe_vectors.npy")
recipe_vectors_pooled = np.load(EMB_DIR / "recipe_vectors_from_ingredients.npy")

print(f"Word2Vec vocab:               {len(w2v_model.wv.key_to_index)}")
print(f"ingredient_vectors:           {ingredient_vectors.shape}")
print(f"recipe_vectors (SBERT):       {recipe_vectors.shape}")
print(f"recipe_vectors_pooled (W2V):  {recipe_vectors_pooled.shape}")
print(f"\nSBERT norm check (should be ~1.0): {np.linalg.norm(recipe_vectors[0]):.4f}")
print(f"Word2Vec norm check (should NOT be 1.0): {np.linalg.norm(ingredient_vectors[0]):.4f}")

Loaded 48005 recipes
Word2Vec vocab:               3023
ingredient_vectors:           (3023, 100)
recipe_vectors (SBERT):       (48005, 384)
recipe_vectors_pooled (W2V):  (48005, 100)

SBERT norm check (should be ~1.0): 1.0000
Word2Vec norm check (should NOT be 1.0): 3.3576


## 1. Ingredient nearest neighbors (Word2Vec)

Hand-picked probes spanning herbs, sauces, fats, fruits, spices, dairy, aromatics, sweeteners, and a more obscure ingredient.

**Expected good results:**
- `basil` → oregano, thyme, parsley, rosemary, sage
- `soy sauce` → sesame oil, rice vinegar, fish sauce, mirin
- `butter` → margarine, shortening, oil

In [4]:
PROBES = [
    "basil", "soy sauce", "butter", "lime", "jalapeno",
    "cumin", "parmesan", "ginger", "vanilla", "tahini",
]

for probe in PROBES:
    # Try the probe verbatim, then with underscore (Agent 1 may have joined multi-word tokens)
    candidates = [probe, probe.replace(" ", "_")]
    found = next((c for c in candidates if c in w2v_model.wv), None)
    if found is None:
        print(f"{probe:14s} -> NOT IN VOCAB (try variations)")
        continue
    neighbors = w2v_model.wv.most_similar(found, topn=8)
    pretty = ", ".join(f"{w} ({s:.2f})" for w, s in neighbors)
    print(f"{found:14s} -> {pretty}")

basil          -> dried basil (0.62), sweet basil (0.59), hot italian sausage (0.58), freshly grnd pepper (0.58), freshly grated parmigiano (0.56), shredded basil (0.55), fresh cheese tortellini (0.55), rigatoni pasta (0.55)
soy sauce      -> soya sauce (0.69), choy (0.69), teriyaki sauce (0.69), hoisin sauce (0.68), shoyu (0.67), hot chili oil (0.66), shaoxing wine (0.66), kelp (0.65)
butter         -> margarine (0.87), oleo (0.72), parkay margarine (0.68), unsalted butter (0.64), evaporated milk (0.61), unbaked pie shell (0.60), hot mashed potato (0.60), corn oil margarine (0.58)
lime           -> whole lime (0.61), fresh squeezed lime juice (0.61), lime zest (0.60), lime wedge (0.59), fresh mango (0.58), jicama (0.58), lime juice (0.57), mango (0.56)
jalapeno       -> jalapeno pepper (0.66), jalapeno chile (0.65), habanero (0.65), jalepeno (0.64), chipotle (0.64), green jalapeno pepper (0.63), cotija cheese (0.63), combine (0.63)
cumin          -> cumin powder (0.70), cumin ground (

## 2. Recipe nearest neighbors (SBERT)

Pick 5 recipes spanning different cuisines/styles and look at their top-5 SBERT neighbors. If a Thai stir-fry's nearest neighbors are all Italian pastas, something is very wrong.

In [5]:
# Pick 5 recipes by index. Adjust these to span cuisines once you've eyeballed the data.
PROBE_RECIPE_IDS = [0, 100, 1000, 5000, 25000]

def show_recipe_neighbors(recipe_id, vectors, label, topn=5):
    sims = vectors @ vectors[recipe_id]
    top_ids = np.argsort(-sims)[:topn + 1]   # +1 because the recipe matches itself
    print(f"\n=== {label}: recipe_id={recipe_id} -- {df.iloc[recipe_id]['title']} ===")
    for rid in top_ids:
        if rid == recipe_id:
            continue
        print(f"  {sims[rid]:.3f}  {df.iloc[rid]['title']}")

for rid in PROBE_RECIPE_IDS:
    show_recipe_neighbors(rid, recipe_vectors, "SBERT", topn=5)


=== SBERT: recipe_id=0 -- Soft Cheesy Pretzel ===
  0.866  Soft Pretzels
  0.858  Soft Pretzels
  0.827  Soft Pretzels
  0.821  Soft Pretzels
  0.815  Soft Pretzels

=== SBERT: recipe_id=100 -- Santa Clara Cookies ===
  0.763  Clara'S Kisses(Forgotten Cookies)
  0.738  Christmas Cookies
  0.731  Saint Barbara Cookies (Lebanese)
  0.714  Christmas Cookies
  0.713  Merry Christmas Cookies

=== SBERT: recipe_id=1000 -- Goose - Roast Port Glazed Goose With Tawny Port Gravy ===
  0.852  Roast Goose with Brandy Cranberry Reduction and Apple Cider Glazed Pearl Onions
  0.806  Roasted Goose with Chestnut-Sweet Potato Stuffing and Gingered Plum Sauce
  0.800  Roast Heirloom Goose With Balsamic Vinegar
  0.773  Roast Goose With Juniper Sauce
  0.760  Spicy Roast Goose With Apple Stuffing

=== SBERT: recipe_id=5000 -- Chocolate Crinkles ===
  0.886  Chocolate Crinkles
  0.853  Chocolate Crinkles
  0.826  Chocolate Crinkles
  0.726  Chocolate-Wrapped Chocolate Cake
  0.722  Chocolate Mounds

=== 

## 3. Vector analogies (Word2Vec)

Classic `king - man + woman = queen` style probes, but on ingredients. These are hit-or-miss — many of the doc's suggested analogies (`butter - baking + frying`) require non-ingredient tokens like `baking` to be in the W2V vocabulary, which they may not be. We handle missing tokens gracefully.

In [6]:
ANALOGIES = [
    # (positive_terms, negative_terms, human-readable description)
    (["butter", "frying"], ["baking"], "butter - baking + frying"),
    (["pasta", "japanese"], ["italian"], "pasta - italian + japanese"),
    (["beef", "indian"],   ["american"], "beef - american + indian"),
    (["basil", "thai"],    ["italian"],  "basil - italian + thai"),
    (["cheddar", "french"],["american"], "cheddar - american + french"),
]

for pos, neg, desc in ANALOGIES:
    missing = [w for w in pos + neg if w not in w2v_model.wv]
    if missing:
        print(f"{desc:40s} -- skipped (missing from vocab: {missing})")
        continue
    try:
        result = w2v_model.wv.most_similar(positive=pos, negative=neg, topn=5)
        pretty = ", ".join(f"{w} ({s:.2f})" for w, s in result)
        print(f"{desc:40s} -> {pretty}")
    except KeyError as e:
        print(f"{desc:40s} -- error: {e}")

butter - baking + frying                 -- skipped (missing from vocab: ['frying'])
pasta - italian + japanese               -> doubanjiang (0.66), chicken soup stock granule (0.64), dashi (0.62), shoyu (0.62), wasabi (0.60)
beef - american + indian                 -- skipped (missing from vocab: ['indian'])
basil - italian + thai                   -- skipped (missing from vocab: ['thai'])
cheddar - american + french              -- skipped (missing from vocab: ['french'])


## 4. SBERT space vs ingredient-pooled space

For the same probe recipe, what are the top neighbors in each space? Hypothesis (worth confirming for the blog): the ingredient-pooled space clusters by **cuisine / ingredient palette**, while SBERT clusters more by **prose style / instruction wording**.

In [7]:
def top_neighbors(recipe_id, vectors, topn=5):
    sims = vectors @ vectors[recipe_id]
    top_ids = np.argsort(-sims)[:topn + 1]
    return [(rid, sims[rid]) for rid in top_ids if rid != recipe_id][:topn]

for rid in PROBE_RECIPE_IDS[:3]:
    print(f"\n=== recipe_id={rid}: {df.iloc[rid]['title']} ===")
    print("  -- SBERT neighbors --")
    for nid, s in top_neighbors(rid, recipe_vectors):
        print(f"    {s:.3f}  {df.iloc[nid]['title']}")
    print("  -- Ingredient-pooled neighbors --")
    for nid, s in top_neighbors(rid, recipe_vectors_pooled):
        print(f"    {s:.3f}  {df.iloc[nid]['title']}")


=== recipe_id=0: Soft Cheesy Pretzel ===
  -- SBERT neighbors --
    0.866  Soft Pretzels
    0.858  Soft Pretzels
    0.827  Soft Pretzels
    0.821  Soft Pretzels
    0.815  Soft Pretzels
  -- Ingredient-pooled neighbors --
    0.979  Quick Cheddar Bread
    0.973  Cheesy Butter Cookie ( a.k.a Lidah Kucing )
    0.962  Cheddar and Cream Scones
    0.961  Scones
    0.961  Mom'S Peach Cobbler

=== recipe_id=100: Santa Clara Cookies ===
  -- SBERT neighbors --
    0.763  Clara'S Kisses(Forgotten Cookies)
    0.738  Christmas Cookies
    0.731  Saint Barbara Cookies (Lebanese)
    0.714  Christmas Cookies
    0.713  Merry Christmas Cookies
  -- Ingredient-pooled neighbors --
    0.949  Special Yellow Cake
    0.940  Butter Cookies(Tea Cakes)
    0.938  Portzelky (New Year'S Cookies)
    0.938  Kabocha Squash Pie with Steamed Kabocha in a Silicone Steamer
    0.934  Doraemon Cupcakes

=== recipe_id=1000: Goose - Roast Port Glazed Goose With Tawny Port Gravy ===
  -- SBERT neighbors --
 

## 5. Findings

_Fill this in after running the cells above. This section is what Agent 4 will lift into the blog post — interesting failures are as valuable as successes._

### What worked
- (e.g. `basil` clusters tightly with other Mediterranean herbs)
- (e.g. SBERT correctly identifies pad thai variants as nearest neighbors)

### What didn't
- (e.g. cuisine analogies fail because cuisine labels aren't in the ingredient vocab — expected)
- (e.g. `tahini` neighbors are noisy — too few co-occurrences in the corpus)

### SBERT vs ingredient-pooled
- (which space clusters cuisine better? give a concrete example)

### Anything broken upstream
- (any tokens like `1/2`, `tbsp`, `of` showing up as 'ingredients'? if so, ping Agent 1)